In [1]:
!wget -O blaze_face_short_range.tflite \
https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite


!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

!pip install mediapipe









--2026-01-15 22:17:23--  https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.135.207, 192.178.163.207, 173.194.202.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.135.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 229746 (224K) [application/octet-stream]
Saving to: ‘blaze_face_short_range.tflite’

blaze_face_short_ra 100%[===================>] 224.36K  --.-KB/s    in 0.002s  

2026-01-15 22:17:23 (118 MB/s) - ‘blaze_face_short_range.tflite’ saved [229746/229746]

Found existing installation: torch 2.9.0+cu126
Uninstalling torch-2.9.0+cu126:
  Successfully uninstalled torch-2.9.0+cu126
Found existing installation: torchvision 0.24.0+cu126
Uninstalling torchvision-0.24.0+cu126:
  Successfully uninstalled torchvision-0.24.0+cu126
Found existing installation: torchaudio 2.9.0+cu126
Uninstall

In [2]:
import torch
import torchvision
print(torch.__version__, torchvision.__version__)
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
import mediapipe as mp
import torch
from torch.utils.data import Dataset
import os



2.5.1+cu121 0.20.1+cu121


In [3]:
import torch
import torch.nn as nn
import torchvision.models as models

class DriverActionClassifier(nn.Module):
    def __init__(self, backbone, num_classes=10):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(3 * 576, num_classes)

    def forward(self, image, face, hand):
        im = self.backbone(image).flatten(1)
        f = self.backbone(face).flatten(1)
        ha = self.backbone(hand).flatten(1)
        combined = torch.cat([im, f, ha], dim=1)
        return self.classifier(combined)



In [5]:
mobilenet = models.mobilenet_v3_small(weights=None)
mobilenet.classifier = nn.Identity()

model = DriverActionClassifier(mobilenet, num_classes=10).cuda()

torch.save(model.state_dict(), "driver_action_deploy.pth")

model.load_state_dict(torch.load("driver_action_deploy.pth"))
model.eval()


/tmp/ipython-input-823785580.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("driver_action_deploy.pth"))


DriverActionClassifier(
  (backbone): MobileNetV3(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
      (1): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
            (activation): ReLU()
            (scale_activation): Hardsigmoid()
          )
       

In [7]:
!pip install onnx onnxruntime


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.3 MB/s eta 0:00:00


In [9]:
dummy_full = torch.randn(1, 3, 224, 224, device="cuda")
dummy_face = torch.randn(1, 3, 224, 224, device="cuda")
dummy_hand = torch.randn(1, 3, 224, 224, device="cuda")

torch.onnx.export(
    model,
    (dummy_full, dummy_face, dummy_hand),
    "driver_action.onnx",
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["full", "face", "hand"],
    output_names=["logits"],
    dynamic_axes={
        "full": {0: "batch"},
        "face": {0: "batch"},
        "hand": {0: "batch"},
        "logits": {0: "batch"},
    }
)

print("ONNX export complete")


ONNX export complete


In [10]:
import onnx
import onnxruntime as ort
import numpy as np

onnx_model = onnx.load("driver_action.onnx")
onnx.checker.check_model(onnx_model)

sess = ort.InferenceSession(
    "driver_action.onnx",
    providers=["CUDAExecutionProvider"]
)

x_full = np.random.randn(2, 3, 224, 224).astype(np.float32)
x_face = np.random.randn(2, 3, 224, 224).astype(np.float32)
x_hand = np.random.randn(2, 3, 224, 224).astype(np.float32)

out = sess.run(
    None,
    {
        "full": x_full,
        "face": x_face,
        "hand": x_hand
    }
)

print(out[0].shape)


(2, 10)


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
